## 데이터 수집 자동화

### 다나와 상품정보 스크래핑

### 자동로그인

- 패스워드 숨기기 
- setx 명령어로 시스템 저장방법

#### 1.  setx 명령어로 시스템에 저장방법
- powershell

``` 
> setx DANAWA_PASSWORD "비밀번호"
```
성공: 지정한 값을 저장했습니다.

```
> setx DANAWA_ID "아이디"
```
성공: 지정한 값을 저장했습니다.

- sysdm.cpl 에서 고급>환경변수 저장된 패스워드 삭제

#### 2. keyring 으로 원도우 자격증명 저장소
- setx와 달리 아예 오픈 안됨
- 패키지 설치

In [1]:
!pip install keyring

Defaulting to user installation because normal site-packages is not writeable


In [2]:
import keyring
from getpass import getpass #password 보안을 위해 사용

In [3]:
username = input('다나와 아이디')
password = getpass('password')

#아이디 저장
keyring.set_password('danawa.com', 'user_id', username)
#패스워드 저장
keyring.set_password('danawa.com', username, password)

print('아이디/패스워드 저장완료')

아이디/패스워드 저장완료


In [4]:
user_id = keyring.get_password('danawa.com', 'user_id')
user_id

''

In [5]:
password = keyring.get_password('danawa.com', user_id)
password is not None

True

In [6]:
import pandas as pd
import os

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

In [7]:
driver = webdriver.Chrome()
wait = WebDriverWait(driver, 10)
driver.get('https://www.danawa.com/')

wait.until(
    EC.presence_of_element_located((By.ID, 'danawa_footer'))
)


<selenium.webdriver.remote.webelement.WebElement (session="18ed9ef31d08d54eadae42a5237123c1", element="f.8CFDCC07019DA49AEC59A62FE49E49B9.d.E66AA6419D47C1D36EE2951A972E899C.e.75")>

In [8]:
# 로그인 링크 확인 후 클릭
driver.find_element(By.LINK_TEXT, '로그인').click()

In [9]:
# 로그인하기
id_input = driver.find_element(By.CSS_SELECTOR, 'input[type="text"]')
id_input.send_keys(user_id)

pass_input = driver.find_element(By.CSS_SELECTOR, 'input[type="password"]')
pass_input.send_keys(password)

In [10]:
login_btn = driver.find_element(By.CLASS_NAME, 'btn_login')
login_btn.click()

#### 동적 페이지 스크래핑

- https://prod.danawa.com/list/?cate=11255834&15main_11_02

In [11]:
driver.get('https://prod.danawa.com/list/?cate=11255834&15main_11_02')

wait.until(
    EC.presence_of_element_located((By.TAG_NAME, 'footer'))
)

<selenium.webdriver.remote.webelement.WebElement (session="18ed9ef31d08d54eadae42a5237123c1", element="f.8CFDCC07019DA49AEC59A62FE49E49B9.d.A7E844C946393C768E5F140A5994F5C6.e.6328")>

- select 태그 페이지 내에 1개만 존재

In [12]:
options = driver.find_elements(By.TAG_NAME, 'option')

for option in options:
    if option.text == '90개':
        option.click()


wait.until(
    EC.presence_of_element_located((By.TAG_NAME, 'footer'))
)

<selenium.webdriver.remote.webelement.WebElement (session="18ed9ef31d08d54eadae42a5237123c1", element="f.8CFDCC07019DA49AEC59A62FE49E49B9.d.A7E844C946393C768E5F140A5994F5C6.e.6328")>

In [ ]:
products = driver.find_elements(By.CSS_SELECTOR, 'div[data-testid="ProductListItem"]')
len(products)

90

In [18]:
product = products[0]
product.text

'1\n영웅컴퓨터 영웅 875 게이밍울트라560X R5 5600 RTX5060\n5600 / RTX 5060 / (AMD) B550 / OS미포함 / 600W / AMD / 라이젠 5000시리즈 / 라이젠5 / 버미어 / DDR4 / 16GB / M.2 / 256GB / NVIDIA / 그래픽 메모리: 8GB / 1Gbps 유선 / HDMI / DP포트 / 파워서플라이 / 미들타워 / LED쿨러 / 용도: 게임용\n혜택 최저가\n1,125,110원\n[옥션] 삼성카드\n/\n무이자 최대 24개월\n기획전\n성능도 가격도 GOOD! 조립PC 인기상품 추천\n21.03. 등록\n브랜드로그\n의견 70\n4.8\n리뷰수\n(230)\n관심\n대량구매\n2위\n16GB, M.2 256GB\n1,209,640\n원\n5몰\nVS상품비교에 추가\nNew 새로워진 상품비교 기능을 이용해보세요.\n16GB, M.2 512GB\n1,259,670\n원\n5몰\nVS상품비교에 추가\n1위\n32GB, M.2 1TB\n1,539,640\n원\n5몰\nVS상품비교에 추가'

In [ ]:
# 1번 인덱스 값에서 제품명, 제품스펙 2번 인덱스 값에서 가격 가져오기
product.find_elements(By.CSS_SELECTOR, '.dnw-product-list-item > div')

[<selenium.webdriver.remote.webelement.WebElement (session="18ed9ef31d08d54eadae42a5237123c1", element="f.8CFDCC07019DA49AEC59A62FE49E49B9.d.A7E844C946393C768E5F140A5994F5C6.e.6824")>,
 <selenium.webdriver.remote.webelement.WebElement (session="18ed9ef31d08d54eadae42a5237123c1", element="f.8CFDCC07019DA49AEC59A62FE49E49B9.d.A7E844C946393C768E5F140A5994F5C6.e.6825")>,
 <selenium.webdriver.remote.webelement.WebElement (session="18ed9ef31d08d54eadae42a5237123c1", element="f.8CFDCC07019DA49AEC59A62FE49E49B9.d.A7E844C946393C768E5F140A5994F5C6.e.6826")>]

In [20]:
computer_info = product.find_elements(By.CSS_SELECTOR, '.dnw-product-list-item > div')[1]
computer_info.text

'1\n영웅컴퓨터 영웅 875 게이밍울트라560X R5 5600 RTX5060\n5600 / RTX 5060 / (AMD) B550 / OS미포함 / 600W / AMD / 라이젠 5000시리즈 / 라이젠5 / 버미어 / DDR4 / 16GB / M.2 / 256GB / NVIDIA / 그래픽 메모리: 8GB / 1Gbps 유선 / HDMI / DP포트 / 파워서플라이 / 미들타워 / LED쿨러 / 용도: 게임용\n혜택 최저가\n1,125,110원\n[옥션] 삼성카드\n/\n무이자 최대 24개월\n기획전\n성능도 가격도 GOOD! 조립PC 인기상품 추천\n21.03. 등록\n브랜드로그\n의견 70\n4.8\n리뷰수\n(230)\n관심\n대량구매'

In [ ]:
# 컴퓨터 이름 출력
computer_info.find_element(By.CSS_SELECTOR, 'div a').text

'영웅컴퓨터 영웅 875 게이밍울트라560X R5 5600 RTX5060'

In [25]:
# 컴퓨터 사양 출력
specs = computer_info.find_elements(By.CSS_SELECTOR, 'div[data-testid="ProductListSpecs"]')

'/'.join(spec.text for spec in specs)

'5600 / RTX 5060 / (AMD) B550 / OS미포함 / 600W / AMD / 라이젠 5000시리즈 / 라이젠5 / 버미어 / DDR4 / 16GB / M.2 / 256GB / NVIDIA / 그래픽 메모리: 8GB / 1Gbps 유선 / HDMI / DP포트 / 파워서플라이 / 미들타워 / LED쿨러 / 용도: 게임용'

In [21]:
price_info = product.find_elements(By.CSS_SELECTOR, '.dnw-product-list-item > div')[2]
price_info.text

'2위\n16GB, M.2 256GB\n1,209,640\n원\n5몰\nVS상품비교에 추가\nNew 새로워진 상품비교 기능을 이용해보세요.\n16GB, M.2 512GB\n1,259,670\n원\n5몰\nVS상품비교에 추가\n1위\n32GB, M.2 1TB\n1,539,640\n원\n5몰\nVS상품비교에 추가'

In [26]:
price_info.find_element(By.CSS_SELECTOR, 'li').text

'2위\n16GB, M.2 256GB\n1,209,640\n원\n5몰\nVS상품비교에 추가\nNew 새로워진 상품비교 기능을 이용해보세요.'

In [28]:
from pathlib import Path
from urllib.parse import urlparse
import requests